# Notebook 13 — Reading Biology Figures

**Module:** 05 — Biology Fundamentals  
**Notebook:** 13 of 18  
**Prerequisites:** NB03, NB09  
**Time estimate:** 60 minutes

> **Track A critical.** 'Tell me what this figure shows' is the most common
> opening question in a paper-reading interview. This notebook trains that skill directly.

---
## Step 1 — Motivation

Bioinformatics papers are dense with specialized figure types: Manhattan plots, volcano
plots, heatmaps, western blots, survival curves, Kaplan-Meier plots, dotplots,
violin plots. Being able to read any of these cold — understand what the axes mean,
what the data shows, and what biological conclusion the authors intend — is a core
professional skill.

---
## Step 3 — Biological Background

**Key figure types in computational biology:**

| Figure type | What it shows | Key features to identify |
|-------------|--------------|-------------------------|
| Manhattan plot | GWAS p-values across genome | Chromosome position (x), -log10(p) (y), genome-wide significance line |
| Volcano plot | DE gene effect vs. significance | log2FC (x), -log10(p) (y), top-right = up-regulated significant |
| Heatmap | Matrix of values (e.g. expression) | Color scale, row/column clustering |
| Western blot | Protein presence/size | Band position = molecular weight; band intensity = abundance |
| Kaplan-Meier | Survival over time | Y drops at each event; p-value from log-rank test |
| PCA plot | Sample clustering in PC space | Clusters = similar expression profiles |
| Box/violin plot | Distribution of a variable | Median, IQR, outliers; violin shows density |
| Dot plot | GO term enrichment | Dot size = gene count; color = p-value |
| ROC curve | Classifier performance | Area under curve (AUC); diagonal = random |

**How to read any biology figure (Three-question protocol):**
1. **What is on each axis?** Units? Scale (log or linear)?
2. **What does one data point represent?** One gene? One patient? One SNP?
3. **What is the biological conclusion?** Not what the data shows — what the authors
   claim it means. Are you convinced?

---
## Step 6 — Python Implementation: Build Common Figure Types

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
# Cell 6.1 — Manhattan plot
rng = np.random.default_rng(42)

# Simulate GWAS summary statistics
n_snps_per_chr = 500
n_chromosomes = 22
chr_lengths = np.array([248, 243, 198, 190, 181, 170, 159, 145, 138, 133,
                         135, 133, 114, 107, 102, 90, 81, 78, 58, 64, 46, 50])

# Generate SNP positions and p-values
all_pos = []
all_chrom = []
all_logp = []

cum_pos = 0
chr_midpoints = []

for chrom_idx, chr_len in enumerate(chr_lengths):
    n = n_snps_per_chr
    positions = rng.uniform(0, chr_len, size=n)
    # Null p-values from uniform distribution
    pvals = rng.uniform(0, 1, size=n)

    # Add a signal on chromosome 6 (simulate HLA association)
    if chrom_idx == 5:
        signal_snps = rng.choice(n, size=10, replace=False)
        pvals[signal_snps] = rng.uniform(1e-9, 1e-7, size=10)

    logp = -np.log10(pvals)
    all_pos.extend(cum_pos + positions)
    all_chrom.extend([chrom_idx + 1] * n)
    all_logp.extend(logp)
    chr_midpoints.append(cum_pos + chr_len / 2)
    cum_pos += chr_len + 5  # add gap between chromosomes

all_pos   = np.array(all_pos)
all_chrom = np.array(all_chrom)
all_logp  = np.array(all_logp)
print(f"Generated {len(all_pos):,} SNPs across {n_chromosomes} chromosomes")
print(f"Most significant: -log10(p) = {all_logp.max():.1f} on chromosome {all_chrom[all_logp.argmax()]}")

In [ ]:
# Cell 6.2 — Kaplan-Meier survival curve (manual implementation)
def kaplan_meier(times: np.ndarray, events: np.ndarray) -> tuple:
    """Compute Kaplan-Meier survival function."""
    sorted_idx = np.argsort(times)
    times = times[sorted_idx]
    events = events[sorted_idx]

    t_unique = np.unique(times[events == 1])
    n = len(times)
    survival = [1.0]
    t_points = [0.0]

    for t in t_unique:
        at_risk = (times >= t).sum()
        n_events = ((times == t) & (events == 1)).sum()
        s_prev = survival[-1]
        survival.append(s_prev * (1 - n_events / at_risk))
        t_points.append(t)

    return np.array(t_points), np.array(survival)

# Simulate two survival groups (high vs. low PRS)
n_per_group = 200
# High risk: shorter median survival
t_high = rng.exponential(scale=36, size=n_per_group)   # median ~25 months
e_high = rng.binomial(1, 0.8, size=n_per_group)         # 80% events observed
# Low risk: longer median survival
t_low  = rng.exponential(scale=60, size=n_per_group)   # median ~42 months
e_low  = rng.binomial(1, 0.8, size=n_per_group)

t_h, s_h = kaplan_meier(t_high, e_high)
t_l, s_l = kaplan_meier(t_low, e_low)
print(f"Kaplan-Meier computed for {n_per_group} patients per group")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Four key figure types
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Manhattan plot
ax = axes[0, 0]
colors_chr = ['steelblue', 'cornflowerblue']
for chrom_idx in range(1, n_chromosomes + 1):
    mask = all_chrom == chrom_idx
    color = colors_chr[(chrom_idx - 1) % 2]
    ax.scatter(all_pos[mask], all_logp[mask], s=2, color=color, alpha=0.5)

gwas_threshold = -np.log10(5e-8)
suggestive = -np.log10(1e-5)
ax.axhline(gwas_threshold, color='tomato', ls='--', lw=1, label=f'Genome-wide sig (5×10⁻⁸)')
ax.axhline(suggestive, color='orange', ls=':', lw=0.8, label='Suggestive (10⁻⁵)')
ax.set_xticks(chr_midpoints[::2])
ax.set_xticklabels([str(i) for i in range(1, n_chromosomes+1, 2)], fontsize=7)
ax.set_xlabel('Chromosome'); ax.set_ylabel('-log10(p)')
ax.set_title('Manhattan plot: GWAS p-values across genome')
ax.legend(fontsize=7)

# Panel 2: Volcano plot
ax = axes[0, 1]
n_genes = 2000
log2fc = rng.normal(0, 1, n_genes)
neg_log10p = rng.exponential(1, n_genes)
# Make some truly significant points
sig_mask = (np.abs(log2fc) > 1.5) & (neg_log10p > 2)
# Color by significance
colors_v = np.where(
    (log2fc > 1.5) & (neg_log10p > 2), 'tomato',
    np.where((log2fc < -1.5) & (neg_log10p > 2), 'steelblue', 'lightgray')
)
ax.scatter(log2fc, neg_log10p, c=colors_v, s=5, alpha=0.6)
ax.axvline(1.5, color='gray', ls='--', lw=0.8); ax.axvline(-1.5, color='gray', ls='--', lw=0.8)
ax.axhline(2, color='gray', ls=':', lw=0.8)
ax.set_xlabel('log2 Fold Change'); ax.set_ylabel('-log10(p-value)')
ax.set_title(f'Volcano plot: {sig_mask.sum()} significant DE genes\n(red=up, blue=down; |FC|>1.5, p<0.01)')

# Panel 3: Heatmap
ax = axes[1, 0]
# 30 genes × 12 samples, 3 clusters
n_g, n_s = 30, 12
expr = rng.normal(0, 0.5, (n_g, n_s))
# Add signal: genes 0-9 high in samples 0-3 (cluster 1)
expr[:10, :4] += 2
# Genes 10-19 high in samples 4-7 (cluster 2)
expr[10:20, 4:8] += 2
# Genes 20-29 high in samples 8-11 (cluster 3)
expr[20:, 8:] += 2

im = ax.imshow(expr, aspect='auto', cmap='RdBu_r', vmin=-3, vmax=3)
plt.colorbar(im, ax=ax, label='log2 expression')
ax.set_xlabel('Sample'); ax.set_ylabel('Gene')
ax.set_title('Expression heatmap: 3 gene clusters × 3 sample groups')
for x_sep in [4, 8]:
    ax.axvline(x_sep - 0.5, color='white', lw=2)
for y_sep in [10, 20]:
    ax.axhline(y_sep - 0.5, color='white', lw=2)

# Panel 4: Kaplan-Meier
ax = axes[1, 1]
ax.step(t_h, s_h, where='post', color='tomato', lw=2, label='High PRS')
ax.step(t_l, s_l, where='post', color='steelblue', lw=2, label='Low PRS')
ax.set_xlabel('Time (months)'); ax.set_ylabel('Survival probability')
ax.set_title('Kaplan-Meier survival curves:\nHigh vs. low polygenic risk score')
ax.legend(); ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='gray', ls=':', lw=0.8, label='Median survival')

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. For the Manhattan plot: what does a SNP above the genome-wide significance line
   (p < 5×10⁻⁸) actually mean? Why is the threshold 5×10⁻⁸ and not 0.05?
2. For the volcano plot: a gene has log2FC = 3 and p = 0.2. Another has log2FC = 0.5
   and p = 10⁻¹⁰. Which is more likely biologically meaningful? Why?
3. Implement `kaplan_meier(times, events)` from scratch. What is the interpretation
   of the y-axis value at time t=24 months?
4. A heatmap shows row clustering where genes in the same cluster have similar
   expression patterns. What does a correlation-based distance metric use to
   cluster genes? How does this differ from Euclidean distance?

---
## Quiz — Active Recall

1. In a volcano plot, what do points in the top-right corner represent?
2. What is the genome-wide significance threshold in GWAS and why that value?
3. In a Kaplan-Meier curve, what does a censored observation represent?
4. What does a western blot band at ~50 kDa tell you?
5. In a PCA plot, samples that cluster together have what in common?

---
## Reflection

**Date completed:** ____________________

> *[Pull up Figure 1 from any recent Nature Genetics paper. Apply the three-question
> protocol: axes, data point, biological conclusion. Can you do it in under 2 minutes?]*

---
*Next: `14_biology_vocabulary_50_terms.ipynb`*